In [1]:
import pandas as pd
import numpy as np
def pick_hours(night, n_hours=4):

    # Sort by signal first, then by time, so a tie is won by the earlier hour
    ranked = night.sort_values(["signal", "hour_start_nem"])
    # Keep the n_hours best rows
    chosen = ranked.head(n_hours)
    # Put them back in time order so the result is easy to read
    return chosen.sort_values("hour_start_nem")

def pick_fixed(night, n_hours=4):
    """Baseline: plug in at 18:00 and charge straight away (no signal)."""
    # The first n_hours of the night in time order = 18:00, 19:00, 20:00, 21:00
    return night.sort_values("hour_start_nem").head(n_hours)

In [2]:
night = pd.DataFrame({
    "hour_start_nem":    pd.date_range("2025-07-01 18:00", periods=13, freq="h"),
    "hour_start_sydney": pd.date_range("2025-07-01 18:00", periods=13, freq="h"),
    #           18    19    20    21    22    23    00    01    02    03    04    05    06
    "signal": [0.50, 0.60, 0.70, 0.80, 0.30, 0.20, 0.10, 0.40, 0.40, 0.90, 0.95, 0.85, 0.75],
})

# By hand: the lowest are 00:00 (0.10), 23:00 (0.20), 22:00 (0.30).
# The 4th place is a tie between 01:00 and 02:00 (both 0.40), so the earliest, 01:00, wins.
expected = [22, 23, 0, 1]

picked = pick_hours(night)
assert list(picked["hour_start_sydney"].dt.hour) == expected, "scheduler picked the wrong hours"
assert len(picked) == 4

fixed = pick_fixed(night)
assert list(fixed["hour_start_sydney"].dt.hour) == [18, 19, 20, 21], "fixed baseline is wrong"

print("All tests passed")

All tests passed


In [3]:
def load_hourly(path):
    """Read a shared hourly table (MEF, AEI or price). Times stay as plain local clock times."""
    return pd.read_csv(path, parse_dates=["hour_start_nem", "hour_start_sydney"])


def expected_hours(night_of):
    """How many real hours the night 18:00 on D to 06:00 on D+1 has (13, or 12 / 14 at daylight saving)."""
    start = (night_of + pd.Timedelta(hours=18)).tz_localize("Australia/Sydney")
    end = (night_of + pd.Timedelta(days=1, hours=6)).tz_localize("Australia/Sydney")
    return int((end - start) / pd.Timedelta(hours=1)) + 1


def slice_nights(df):
    """Keep only hours inside the 18:00-06:00 window, label each with night_of, keep complete nights."""
    hour = df["hour_start_sydney"].dt.hour
    df = df[(hour >= 18) | (hour <= 6)].copy()
    # Subtract 18 h: 18:00 on D -> 00:00 on D, 06:00 on D+1 -> 12:00 on D. Then take the date.
    df["night_of"] = (df["hour_start_sydney"] - pd.Timedelta(hours=18)).dt.normalize()
    counts = df.groupby("night_of")["hour_start_nem"].transform("count")
    expected = df["night_of"].map(expected_hours)
    return df[counts == expected].sort_values("hour_start_nem")

In [4]:
# Fake year in NEM time, then convert to Sydney time 
nem = pd.date_range("2025-07-01 00:00", "2026-06-30 23:00", freq="h")
fake = pd.DataFrame({"hour_start_nem": nem})
fake["hour_start_sydney"] = (nem.tz_localize("Australia/Brisbane")
                                .tz_convert("Australia/Sydney").tz_localize(None))
fake["signal"] = np.random.rand(len(fake))

nights = slice_nights(fake)
sizes = nights.groupby("night_of").size()

assert len(fake) == 8760
assert len(sizes) == 364, len(sizes)                      # 1 Jul 2025 to 29 Jun 2026
assert sizes.loc["2025-10-04"] == 12 and sizes.loc["2026-04-04"] == 14
assert (sizes.drop([pd.Timestamp("2025-10-04"), pd.Timestamp("2026-04-04")]) == 13).all()

# Run the Step 1 picker on every night
for d, night in nights.groupby("night_of"):
    assert len(pick_hours(night)) == 4
    assert list(pick_fixed(night)["hour_start_sydney"].dt.hour) == [18, 19, 20, 21]
print("Step 2 tests passed")

Step 2 tests passed


In [5]:
price = slice_nights(load_hourly("price_hourly_nsw1_v1.csv"))
aei = slice_nights(load_hourly("aei_hourly_nsw1_v1.csv"))

print("price nights:", price["night_of"].nunique())   # expect 364
print("aei nights:  ", aei["night_of"].nunique())     # expect 364

# Hours per night: expect 13 almost everywhere, 12 on 2025-10-04, 14 on 2026-04-04
print(price.groupby("night_of").size().value_counts())

price nights: 364
aei nights:   364
13    362
12      1
14      1
Name: count, dtype: int64


In [6]:
KWH_PER_HOUR = 9.2 / 4   # 2.3 kWh in each of the 4 charging hours

# Join price and AEI on hour_start_nem (the unique key)
p = price[["hour_start_nem", "hour_start_sydney", "night_of", "value"]].rename(columns={"value": "price"})
a = aei[["hour_start_nem", "value"]].rename(columns={"value": "aei"})
hours = p.merge(a, on="hour_start_nem", how="inner")

# Missing-night rule: drop any night with an empty price or AEI value
bad_nights = hours.loc[hours[["price", "aei"]].isna().any(axis=1), "night_of"].unique()
hours = hours[~hours["night_of"].isin(bad_nights)]
print("nights skipped:", len(bad_nights))


def run_strategy(hours, name, signal_col=None):
    """Pick 4 hours per night by signal_col (or fixed if None), then score cost and AEI."""
    rows = []
    for d, night in hours.groupby("night_of"):
        if signal_col is None:
            chosen = pick_fixed(night)
        else:
            chosen = pick_hours(night.assign(signal=night[signal_col]))
        rows.append({
            "night_of": d,
            "strategy": name,
            "hours_chosen": ";".join(str(h) for h in chosen["hour_start_sydney"].dt.hour),
            "energy_kwh": KWH_PER_HOUR * len(chosen),
            "cost_aud": (KWH_PER_HOUR * chosen["price"]).sum() / 1000,
            "aei_accounted_kg": (KWH_PER_HOUR * chosen["aei"]).sum(),
        })
    return pd.DataFrame(rows)


results = pd.concat([
    run_strategy(hours, "aei", "aei"),
    run_strategy(hours, "price", "price"),
    run_strategy(hours, "fixed", None),
])

assert np.allclose(results["energy_kwh"], 9.2)   # sanity check: same energy for every strategy
print(results.groupby("strategy")[["cost_aud", "aei_accounted_kg"]].sum().round(1))
print(results.head(6))

nights skipped: 0
          cost_aud  aei_accounted_kg
strategy                            
aei          388.1            2135.8
fixed        426.4            2198.1
price        250.2            2267.2
    night_of strategy hours_chosen  energy_kwh  cost_aud  aei_accounted_kg
0 2025-07-01      aei     20;1;2;3         9.2  0.977594          5.370741
1 2025-07-02      aei  18;19;20;21         9.2  1.847564          6.054188
2 2025-07-03      aei  18;19;20;21         9.2  2.254812          6.131758
3 2025-07-04      aei  18;19;20;21         9.2  1.887561          6.231401
4 2025-07-05      aei     22;0;1;2         9.2  0.674966          6.337890
5 2025-07-06      aei  18;19;20;21         9.2  1.511071          6.318563


In [7]:
# 1. Join price, AEI and the three MEF versions on hour_start_nem
hours = price[["hour_start_nem", "hour_start_sydney", "night_of", "value"]].rename(columns={"value": "price"})
hours = hours.merge(aei[["hour_start_nem", "value"]].rename(columns={"value": "aei"}), on="hour_start_nem")
for v in ["B", "C", "Cfwd"]:
    m = load_hourly(f"mef_hourly_{v}_rules-v1.csv")[["hour_start_nem", "value"]]
    hours = hours.merge(m.rename(columns={"value": f"mef_{v}"}), on="hour_start_nem")

# 2. Missing-night rule: skip any night with an empty value
check = ["price", "aei", "mef_B", "mef_C", "mef_Cfwd"]
bad = hours.loc[hours[check].isna().any(axis=1), "night_of"].unique()
hours = hours[~hours["night_of"].isin(bad)]
print("nights skipped:", len(bad), "| nights used:", hours["night_of"].nunique())


def run_strategy(hours, name, signal_col=None):
    """Pick 4 hours per night, then score with the ACTUAL MEF (B, C, C-fwd), price and AEI."""
    rows = []
    for d, night in hours.groupby("night_of"):
        if signal_col is None:
            chosen = pick_fixed(night)
        else:
            chosen = pick_hours(night.assign(signal=night[signal_col]))
        k = KWH_PER_HOUR
        rows.append({
            "night_of": d,
            "strategy": name,
            "hours_chosen": ";".join(str(h) for h in chosen["hour_start_sydney"].dt.hour),
            "energy_kwh": k * len(chosen),
            "cost_aud": (k * chosen["price"]).sum() / 1000,
            "emissions_kg_B": (k * chosen["mef_B"]).sum(),
            "emissions_kg_C": (k * chosen["mef_C"]).sum(),
            "emissions_kg_Cfwd": (k * chosen["mef_Cfwd"]).sum(),
            "aei_accounted_kg": (k * chosen["aei"]).sum(),
            "_set": frozenset(chosen["hour_start_nem"]),   # for the overlap check only
        })
    return pd.DataFrame(rows)


# 3. Run the four strategies (mef_oracle picks by version C)
res = {name: run_strategy(hours, name, col) for name, col in
       [("mef_oracle", "mef_C"), ("aei", "aei"), ("price", "price"), ("fixed", None)]}
results = pd.concat(res.values())
assert np.allclose(results["energy_kwh"], 9.2)          # same energy for every strategy

# 4. The three numbers for the meeting
cols = ["cost_aud", "emissions_kg_B", "emissions_kg_C", "emissions_kg_Cfwd", "aei_accounted_kg"]
print(results.groupby("strategy")[cols].sum().round(1).to_string())
for s in ["aei", "price"]:
    same = (res[s]["_set"].values == res["mef_oracle"]["_set"].values).mean()
    print(f"{s} picks the same 4 hours as mef_oracle on {100*same:.1f}% of nights")

# 5. Save the per-night file
results.drop(columns="_set").to_csv("schedule_results_v1.csv", index=False)

nights skipped: 0 | nights used: 364
            cost_aud  emissions_kg_B  emissions_kg_C  emissions_kg_Cfwd  aei_accounted_kg
strategy                                                                                 
aei            388.1          1904.8          1952.4             2151.4            2135.8
fixed          426.4          1646.3          1696.3             2072.7            2198.1
mef_oracle     368.9          1448.8          1345.0             1722.5            2262.0
price          250.2          2321.7          2274.1             2306.7            2267.2
aei picks the same 4 hours as mef_oracle on 3.0% of nights
price picks the same 4 hours as mef_oracle on 0.0% of nights
